# 14 — Tactical view and offside on in-domain stills (Phase 5, measured)

The notebook 13 video uses a phone clip from a low stand angle, which the keypoint model was never trained on; its minimap is visibly wrong and mostly measures that domain shift. This notebook measures Phase 5 **in domain**: broadcast-camera frames from the detection **test** split that also carry pitch-keypoint labels. There are 16 such frames (videos 121364, 744b27), and all of them sit in the pitch **test** split, so neither model trained on them.

For each frame the offside line is computed twice with identical roles (teams, defending goal, ball, feet):
- **reference line** — homography fitted to the labelled keypoints (label-only fit residual ≈ 0.2 m, notebook 12);
- **automatic line** — homography from the keypoint model (the Phase 4 pipeline).

The gap between them is the offside error caused by automatic calibration, in metres. The reference is not offside ground truth (feet are still box bottom-centres; only visible players count); it isolates the calibration error.

**Keypoint confidence threshold chosen on the pitch val split** (lowest median reprojection error with ≥ 90 % frames calibrated), then used here. Teams: KMeans on jersey colour per video (stills have no tracks). Defending goal from the goalkeeper; defending team = the team nearer that goal on average per video (assumption, as in 13).

In [ ]:
DET_WEIGHTS, DET_IMGSZ, DET_CONF = "/content/results/res_yolo11s_1280/best.pt", 1280, 0.25
KP_WEIGHTS, KP_IMGSZ = "/content/results/pitch_kp_yolo11s/best.pt", 960
CONF_GRID = [0.3, 0.5, 0.7, 0.85]
OUT = "/content/results/tactical_stills"

In [ ]:
import json as _json, pathlib as _pl
_choice = _pl.Path("/content/results/pitch_kp_choice.json")
if _choice.exists():  # keypoint model chosen on val by eval_kp_val.py
    _c = _json.loads(_choice.read_text()); KP_WEIGHTS, KP_IMGSZ = _c["weights"], _c["imgsz"]
print("keypoint model:", KP_WEIGHTS, KP_IMGSZ)
import sys, json, pathlib, collections
sys.path.insert(0, "/content/ballhawk")
import numpy as np, pandas as pd, cv2
from sklearn.cluster import KMeans
from ultralytics import YOLO
import ballhawk_common as bc, ballhawk_pitch as bp, ballhawk_video as bv

out = pathlib.Path(OUT); out.mkdir(parents=True, exist_ok=True)
det_yaml = bc.prepare_data()
pitch_yaml = bc.prepare_data(bc.PITCH_DATA_DIR, "splits_pitch.json", bc.PITCH_RAW_DIR)
det, kp = YOLO(DET_WEIGHTS), YOLO(KP_WEIGHTS)
names = [det.names[i] for i in range(len(det.names))]
PLAYER, KEEPER, REF, BALL = (names.index(n) for n in ("player", "goalkeeper", "referee", "ball"))

def labels(img):
    """Labelled keypoints of a pitch-dataset image in pixels: (visible indices, xy)."""
    h, w = cv2.imread(str(img)).shape[:2]
    k = np.array(open(img.parents[1] / "labels" / f"{img.stem}.txt").read().split(), float)[5:].reshape(-1, 3)
    vis = np.flatnonzero(k[:, 2] > 0)
    return vis, k[vis, :2] * [w, h]

In [ ]:
# Choose the keypoint confidence threshold on the pitch VAL split.
rows = []
for img in bc.split_images(pitch_yaml, "val"):
    vis, gt = labels(img)
    k = bp.predict_keypoints(kp, str(img), KP_IMGSZ)
    for c in CONF_GRID:
        fit = bp.keypoint_homography(k, c)
        err = float(np.median(bp.reprojection_error(fit[0], gt, bp.PITCH_XY[vis])[0])) if fit else np.nan
        rows.append({"conf": c, "calibrated": fit is not None, "median_reproj_m": err})
sweep = pd.DataFrame(rows).groupby("conf").agg(calibrated=("calibrated", "mean"), median_reproj_m=("median_reproj_m", "median"))
ok = sweep[sweep.calibrated >= 0.9]
KP_CONF = float((ok if len(ok) else sweep).median_reproj_m.idxmin())
print(sweep.round(3)); print("chosen on val: KP_CONF =", KP_CONF)

In [ ]:
# Frames in both datasets: detection test images whose stem has pitch-keypoint labels.
pitch_by_stem = {p.name.split(".rf.")[0]: p for s in ("train", "val", "test") for p in bc.split_images(pitch_yaml, s)}
frames = [(img, pitch_by_stem[img.name.split(".rf.")[0]]) for img in bc.split_images(det_yaml, "test")
          if img.name.split(".rf.")[0] in pitch_by_stem]
pitch_test = {p.name for p in bc.split_images(pitch_yaml, "test")}
assert all(p.name in pitch_test for _, p in frames), "shared frames must be in the pitch test split"
print(len(frames), "shared frames, all in the pitch test split")

# Detections, jersey colours, both homographies.
recs = []
for img, pimg in frames:
    im = cv2.imread(str(img))
    b = det.predict(im, imgsz=DET_IMGSZ, conf=DET_CONF, verbose=False)[0].boxes
    xyxy, cls = b.xyxy.cpu().numpy(), b.cls.int().cpu().numpy()
    vis, gt = labels(pimg)
    H_ref, _ = bp.fit_homography(gt, bp.PITCH_XY[vis])
    fit = bp.keypoint_homography(bp.predict_keypoints(kp, im, KP_IMGSZ), KP_CONF)
    grass = bv.grass_lab(im)
    colours = [bv.jersey_colour(im, bx, grass) if c == PLAYER else None for bx, c in zip(xyxy, cls)]
    recs.append({"img": img, "video": img.name.split("_")[0], "xyxy": xyxy, "cls": cls, "conf": b.conf.cpu().numpy(),
                 "colours": colours, "H_ref": H_ref, "H_auto": None if fit is None else fit[0]})

# Teams per video (KMeans k=2 on jersey colour); defending goal and team from the reference homography.
for v in {r["video"] for r in recs}:
    rv = [r for r in recs if r["video"] == v]
    cols = [(r_i, j, c) for r_i, r in enumerate(rv) for j, c in enumerate(r["colours"]) if c is not None]
    lab = KMeans(2, n_init=10, random_state=0).fit_predict(np.array([c for *_, c in cols]))
    for r in rv:
        r["team"] = {}
    for (r_i, j, _), t in zip(cols, lab):
        rv[r_i]["team"][j] = int(t)
    keeper_x = [bp.project(r["H_ref"], bp.foot_points(r["xyxy"][r["cls"] == KEEPER]))[:, 0] for r in rv]
    keeper_x = np.concatenate(keeper_x)
    goal = bp.defending_goal_x(float(np.median(keeper_x))) if len(keeper_x) else None
    depth = collections.defaultdict(list)
    for r in rv:
        P = bp.project(r["H_ref"], bp.foot_points(r["xyxy"]))
        for j, t in r["team"].items():
            depth[t].append(abs(P[j, 0] - goal)) if goal is not None else None
    defending = min(depth, key=lambda t: np.mean(depth[t])) if depth and goal is not None else None
    for r in rv:
        r["goal"], r["defending"] = goal, defending
    print(f"video {v}: {len(rv)} frames, defending goal x={goal}, defending team={defending}")

In [ ]:
def offside_with(r, H):
    P = bp.project(H, bp.foot_points(r["xyxy"]))
    role = ["def" if (c == KEEPER or r["team"].get(j) == r["defending"]) and c != REF and c != BALL
            else "att" if j in r["team"] else "ball" if c == BALL else "other" for j, c in enumerate(r["cls"])]
    on = [(0 <= p[0] <= bp.LENGTH and 0 <= p[1] <= bp.WIDTH) for p in P]
    dx = [p[0] for p, ro, o in zip(P, role, on) if o and ro == "def"]
    att = [j for j, (ro, o) in enumerate(zip(role, on)) if o and ro == "att"]
    balls = [j for j, ro in enumerate(role) if ro == "ball"]
    bx = P[max(balls, key=lambda j: r["conf"][j]), 0] if balls else None
    line, flags = bp.offside(dx, P[att, 0] if att else [], r["goal"], bx)
    return line, dict(zip(att, flags)), P

rows = []
for r in recs:
    if r["goal"] is None or r["defending"] is None:
        rows.append({"image": r["img"].name, "status": "no goalkeeper -> no defending side"}); continue
    line_ref, flags_ref, P_ref = offside_with(r, r["H_ref"])
    if r["H_auto"] is None:
        rows.append({"image": r["img"].name, "status": "automatic calibration failed"}); continue
    line_auto, flags_auto, P_auto = offside_with(r, r["H_auto"])
    players = [j for j in r["team"]]
    r.update(line_ref=line_ref, line_auto=line_auto, P_ref=P_ref, P_auto=P_auto, flags_ref=flags_ref, flags_auto=flags_auto)
    rows.append({"image": r["img"].name, "status": "ok",
                 "line_ref_x": line_ref, "line_auto_x": line_auto,
                 "line_error_m": abs(line_auto - line_ref) if (line_ref is not None and line_auto is not None) else np.nan,
                 "player_pos_error_m": float(np.median(np.linalg.norm(P_auto[players] - P_ref[players], axis=1))) if players else np.nan,
                 "flags_ref": int(sum(flags_ref.values())), "flags_auto": int(sum(flags_auto.values())),
                 "flag_decisions_differ": int(sum(flags_ref.get(j, False) != flags_auto.get(j, False) for j in set(flags_ref) | set(flags_auto)))})
df = pd.DataFrame(rows); df.to_csv(out / "frames.csv", index=False)
okdf = df[df.status == "ok"]
summary = {"frames": len(df), "evaluated": int(len(okdf)), "kp_conf_chosen_on_val": KP_CONF,
           "median_line_error_m": float(okdf.line_error_m.median()), "p90_line_error_m": float(okdf.line_error_m.quantile(0.9)),
           "median_player_pos_error_m": float(okdf.player_pos_error_m.median()),
           "frames_with_a_flag_decision_changed": int((okdf.flag_decisions_differ > 0).sum()),
           "statuses": df.status.value_counts().to_dict()}
print(df.round(2).to_string()); print(json.dumps(summary, indent=2))

In [ ]:
# Figure: 4 frames — image with reference (white) and automatic (blue) lines, and the minimap with both.
show = [r for r in recs if "line_ref" in r and r["line_ref"] is not None and r["line_auto"] is not None][:4]
tiles = []
for r in show:
    im = cv2.imread(str(r["img"]))
    im = bp.draw_offside_line(im, r["H_ref"], r["line_ref"], colour=(255, 255, 255), alpha=0.5)
    im = bp.draw_offside_line(im, r["H_ref"], r["line_auto"], colour=(214, 120, 42), alpha=0.5)
    mm, to_px = bp.minimap(scale=8)
    for x, c in ((r["line_ref"], (255, 255, 255)), (r["line_auto"], (214, 120, 42))):
        cv2.line(mm, to_px(x, 0), to_px(x, bp.WIDTH), c, 2)
    for j, t in r["team"].items():
        col = bv.TEAM_BGR[t]
        cv2.circle(mm, to_px(*r["P_ref"][j]), 5, col, -1, cv2.LINE_AA)       # reference position: filled
        cv2.circle(mm, to_px(*r["P_auto"][j]), 6, col, 1, cv2.LINE_AA)       # automatic position: ring
    left = cv2.resize(im, (960, 540)); right = cv2.resize(mm, (int(mm.shape[1] * 540 / mm.shape[0]), 540))
    tile = np.hstack([left, right])
    cv2.putText(tile, f"{r['img'].name.split('.rf.')[0]}  line error {abs(r['line_auto'] - r['line_ref']):.2f} m",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    tiles.append(tile)
if tiles:
    cv2.imwrite(str(out / "offside_ref_vs_auto.jpg"), np.vstack(tiles), [cv2.IMWRITE_JPEG_QUALITY, 85])
summary["legend"] = "white = reference (labelled keypoints), blue = automatic; minimap: filled = reference position, ring = automatic"
(out / "summary.json").write_text(json.dumps(summary, indent=2))
print(out / "offside_ref_vs_auto.jpg")